# From Notebook to Production: ML Model Deployment

Learn how to take a trained model and deploy it as a web API, containerize it, and set up monitoring. This bridges the gap between Jupyter notebooks and real-world production systems.


In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

print('Ready for production ML!')


## 1. Train a Production-Ready Model


In [ ]:
df = pd.read_csv('data/customer_churn.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
# Build a clean preprocessing + model pipeline
categorical = ['contract_type', 'internet_service', 'payment_method']
numeric = ['tenure_months', 'monthly_charges', 'total_charges']

X = df[categorical + numeric]
y = df['churn']

# One-hot encode categoricals
X_encoded = pd.get_dummies(X, columns=categorical, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


## 2. Serialize the Model (Pickle)


In [ ]:
# Save model to disk
with open('model.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print('Model saved as model.pkl')

# Load and verify
with open('model.pkl', 'rb') as f:
    loaded = pickle.load(f)
print(f'Loaded model: {type(loaded).__name__}')
print(f'Test accuracy: {loaded.score(X_test, y_test):.3f}')


## 3. Create an API Server (FastAPI Style)


In [ ]:
api_code = """
# app.py — FastAPI ML API
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import pandas as pd

app = FastAPI(title='Churn Prediction API')

# Load model
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

class CustomerData(BaseModel):
    tenure_months: float
    monthly_charges: float
    total_charges: float
    contract_type: str
    internet_service: str
    payment_method: str

@app.post('/predict')
def predict(data: CustomerData):
    df = pd.DataFrame([data.model_dump()])
    prob = model.predict_proba(df)[0, 1]
    pred = int(prob > 0.5)
    return {'prediction': pred, 'probability': round(prob, 4)}

@app.get('/health')
def health():
    return {'status': 'ok'}

# Run with: uvicorn app:app --reload
"""
print(api_code)


## 4. Test the API


In [ ]:
test_request = """
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"tenure_months": 12, "monthly_charges": 89.90,
       "total_charges": 1078.80, "contract_type": "Month-to-month",
       "internet_service": "Fiber optic", "payment_method": "Electronic check"}'
"""
print('Run this in terminal:')
print(test_request)
print('\nExpected response: {"prediction": 1, "probability": 0.8732}')


## 5. Docker Containerization


In [ ]:
dockerfile = """
# Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py model.pkl .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print('Dockerfile:')
print(dockerfile)
print('\nBuild: docker build -t churn-api .')
print('Run:   docker run -p 8000:8000 churn-api')


## 6. Model Monitoring Checklist


In [ ]:
print('''
Production ML Checklist:
------------------------
[ ] Data validation (schema checks, distribution monitoring)
[ ] Model versioning (MLflow, DVC, or simple git tags)
[ ] Prediction logging (store inputs, outputs, timestamps)
[ ] Performance monitoring (track accuracy over time)
[ ] Retraining trigger (drift detection or scheduled retraining)
[ ] A/B testing framework for model comparison
[ ] Rollback strategy (keep previous model versions)
[ ] CI/CD pipeline for automated testing & deployment
''')


## Summary


In [ ]:
print('''
Deployment Options:
- REST API: FastAPI / Flask (simplest, good for internal tools)
- Batch: Scheduled jobs (good for large-scale scoring)
- Streaming: Kafka + ML (real-time, low latency)
- Edge: ONNX / TensorRT (mobile, IoT devices)

Key takeaway: A model in a notebook has zero value.
A model behind an API creates real business impact.
''')
